# SatQuery AI — Module M1: EarthDial GPU Inference Server

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Vishal10052006/SatQuery-AI/blob/M1/m1_earthdial/colab/EarthDial_Colab_Inference_Server.ipynb)

This notebook runs the official **EarthDial 4B RGB VLM** (`akshaydudhane/EarthDial_4B_RGB`, CVPR 2025) on a free **Google Colab Tesla T4 GPU (16 GB VRAM)** and exposes a secure HTTP API endpoint for your local **SatQuery AI M1/M4 modules**.

---
### Step 0: Ensure Free T4 GPU is Selected
1. In the top menu bar, click: **Runtime** > **Change runtime type**
2. Set **Hardware accelerator** to: **T4 GPU**
3. Click **Save**
4. In the top-right corner, click **Connect**

In [ ]:
# Step 1: Verify Tesla T4 GPU Availability
!nvidia-smi

In [ ]:
# Step 2: Install required packages for EarthDial and FastAPI server
# Note: PyTorch & Torchvision are pre-installed in Colab with CUDA support.
!pip install -q transformers==4.37.2 tokenizers==0.15.1 sentencepiece einops timm==0.9.12 accelerate peft bitsandbytes pillow pydantic fastapi uvicorn nest-asyncio python-multipart

In [ ]:
# Step 3: Download and Load EarthDial Model into GPU Memory (BF16)
import torch
from transformers import AutoTokenizer, AutoModel
from torchvision import transforms
from PIL import Image
import io, base64

MODEL_ID = "akshaydudhane/EarthDial_4B_RGB"
print(f"[*] Downloading & loading model checkpoint: {MODEL_ID}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True, use_fast=False)

model = AutoModel.from_pretrained(
    MODEL_ID,
    low_cpu_mem_usage=True,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True
).eval()

image_size = getattr(model.config, "force_image_size", None) or \
             getattr(model.config.vision_config, "image_size", 448)

transform = transforms.Compose([
    transforms.Resize((image_size, image_size), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
vram_used = torch.cuda.memory_allocated(0) / (1024**3) if torch.cuda.is_available() else 0
print(f"[+] EarthDial model successfully loaded on {device_name}!")
print(f"[+] GPU VRAM allocated: {vram_used:.2f} GB")

In [ ]:
# Step 4: Define FastAPI App & Endpoints
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from fastapi.middleware.cors import CORSMiddleware

app = FastAPI(
    title="SatQuery AI - EarthDial GPU Server",
    version="1.0.0",
    description="Google Colab Tesla T4 GPU Inference Server for EarthDial 4B RGB"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class AnalyzeRequest(BaseModel):
    image_base64: str = Field(..., description="Base64-encoded satellite image")
    question: str = Field(..., description="Natural-language question")
    num_beams: int = Field(default=5)
    temperature: float = Field(default=0.0)
    max_new_tokens: int = Field(default=128)

@app.get("/health")
def health():
    return {
        "status": "healthy",
        "model": MODEL_ID,
        "cuda": torch.cuda.is_available(),
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None",
    }

@app.post("/analyze")
def analyze(req: AnalyzeRequest):
    # 1. Decode base64 image
    try:
        img_bytes = base64.b64decode(req.image_base64)
        img = Image.open(io.BytesIO(img_bytes)).convert("RGB")
    except Exception as e:
        raise HTTPException(status_code=400, detail=f"Invalid base64 image: {str(e)}")

    # 2. Transform image tensor to match model device & dtype
    pixel_values = transform(img).unsqueeze(0).to(device=model.device, dtype=model.dtype)

    generation_config = {
        "num_beams": req.num_beams,
        "max_new_tokens": req.max_new_tokens,
        "min_new_tokens": 1,
        "do_sample": req.temperature > 0.0,
        "temperature": req.temperature if req.temperature > 0.0 else 1.0,
    }

    # 3. Execute VLM inference
    try:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        with torch.no_grad():
            answer = model.chat(
                tokenizer=tokenizer,
                pixel_values=pixel_values,
                question=req.question,
                generation_config=generation_config,
                verbose=False
            )

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        return {
            "answer": str(answer).strip(),
            "model": MODEL_ID
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"EarthDial model inference failed: {str(e)}")

In [ ]:
# Step 5: Start Server with Public Cloudflare Tunnel
# 1. Download cloudflared if not already present
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

import subprocess, time, os, re
import nest_asyncio
import uvicorn

# Apply nest_asyncio so uvicorn runs smoothly inside Colab Jupyter event loop
nest_asyncio.apply()

print("[*] Launching Cloudflare tunnel in background...")
tunnel_log = "tunnel.log"
if os.path.exists(tunnel_log):
    os.remove(tunnel_log)

tunnel_cmd = "./cloudflared-linux-amd64 tunnel --url http://127.0.0.1:8000 > tunnel.log 2>&1 &"
os.system(tunnel_cmd)

public_url = None
start_time = time.time()
while time.time() - start_time < 30:
    time.sleep(1.5)
    if os.path.exists(tunnel_log):
        with open(tunnel_log, "r", errors="ignore") as f:
            content = f.read()
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
            if match:
                public_url = match.group(0)
                break

if public_url:
    print("\n" + "=" * 68)
    print("🚀 EARTHDIAL GPU SERVER IS LIVE & READY!")
    print(f"Public Tunnel URL: {public_url}")
    print("\nCopy & paste these commands into your local laptop terminal (PowerShell):")
    print(f'$env:EARTHDIAL_API_URL="{public_url}"')
    print('$env:EARTHDIAL_BACKEND="remote"')
    print("\nThen run inference:")
    print('.\\.venv\\Scripts\\python.exe -m m1_earthdial.inference --image m1_earthdial/examples/sample_satellite.jpg --question \"What can you see in this satellite image?\"')
    print("=" * 68 + "\n")
else:
    print("[-] Could not auto-detect Cloudflare tunnel URL within 30s.")
    print("Check tunnel log below:")
    if os.path.exists(tunnel_log):
        with open(tunnel_log, "r", errors="ignore") as f:
            print(f.read())

# Start FastAPI server on port 8000 (blocks this cell while serving requests)
uvicorn.run(app, host="0.0.0.0", port=8000)